In [0]:
%sql
-- Quantos usuários tem no loyalty?
SELECT
    COUNT(DISTINCT IdCliente)
FROM workspace.tmw_loyalty.clientes ;


In [0]:
%sql
-- Como é o histórico de transacoes diárias? E mensal? E anual?
-- E no lugar de transações, usuário? Quantos usuários ativos distintos a gente tem por mes? E por ano?
-- DIA
SELECT 
    SUBSTR(DtCriacao, 0, 11) AS DtDia, 
    COUNT(*) AS qtdeTransacoes,
    COUNT(DISTINCT idCliente) AS qtdeClientes,
    SUM(QtdePontos)
FROM workspace.tmw_loyalty.transacoes
GROUP BY ALL
ORDER BY DtDia


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Como é o histórico de transacoes diárias? E mensal? E anual?
-- E no lugar de transações, usuário? Quantos usuários ativos distintos a gente tem por mes? E por ano?
-- MÊS
SELECT 
    SUBSTR(DtCriacao, 0, 7) AS DtMes, 
    COUNT(*) AS qtdeTransacoes,
    COUNT(DISTINCT idCliente) AS qtdeClientes,
    SUM(QtdePontos)
FROM workspace.tmw_loyalty.transacoes
GROUP BY ALL
ORDER BY DtMes

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Como é o histórico de transacoes diárias? E mensal? E anual?
-- E no lugar de transações, usuário? Quantos usuários ativos distintos a gente tem por mes? E por ano?
-- ANO
SELECT 
    SUBSTR(DtCriacao, 0, 4) AS DtAno, 
    COUNT(*) AS qtdeTransacoes,
    COUNT(DISTINCT idCliente) AS qtdeClientes,
    SUM(QtdePontos)
FROM workspace.tmw_loyalty.transacoes
GROUP BY ALL
ORDER BY DtAno

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Em 2026 quantos novos usuários?
WITH 
Clientes AS (
    SELECT 
        idCliente,
        SUBSTR(DtCriacao, 0, 11) AS DtDia, 
        SUBSTR(DtCriacao, 0, 4) AS DtAno, 
        COUNT(*) AS qtdeTransacoes,
        SUM(QtdePontos)
    FROM workspace.tmw_loyalty.transacoes
    GROUP BY ALL
    ORDER BY idCliente, DtDia
) ,

Clients_Ano AS (
    SELECT 
        idCliente,
        -- MIN(DtDia) AS Dia_min,
        -- MAX(DtDia) AS Dia_max
        SUBSTR( MIN(DtDia) , 0, 4 ) AS Ano_min    
    FROM Clientes
    GROUP BY ALL
)

SELECT
    Ano_min,
    COUNT(idCliente)
FROM Clients_Ano
-- WHERE Ano_min = '2026'
GROUP BY ALL



In [0]:
%sql
-- Qual produto mais transacionado? E o de maiores valores acumulados?
WITH
transacao_produto AS (
    SELECT 
        t1.*,
        t2.IdProduto,
        t2.QtdeProduto,
        t2.vlProduto,
        t3.DescNomeProduto

    FROM workspace.tmw_loyalty.transacoes AS t1
    LEFT JOIN workspace.tmw_loyalty.transacao_produto AS t2
        ON t1.IdTransacao = t2.IdTransacao 
    LEFT JOIN workspace.tmw_loyalty.produtos AS t3
        ON t2.IdProduto = t3.IdProduto
)

SELECT 
    DescNomeProduto,
    COUNT(*) AS qtdeTransacoes,
    COUNT(DISTINCT IdCliente) AS qtdeClientes,
    SUM(QtdeProduto) AS QtdeProduto,
    SUM(vlProduto) AS vlProduto
FROM transacao_produto
GROUP BY ALL
ORDER BY qtdeTransacoes DESC


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Quando temos picos de transação? E de novos usuários?
WITH 
Clientes AS (
    SELECT 
        idCliente,
        SUBSTR(DtCriacao, 0, 11) AS DtDia, 
        SUBSTR(DtCriacao, 0, 4) AS DtAno, 
        COUNT(*) AS qtdeTransacoes,
        SUM(QtdePontos)
    FROM workspace.tmw_loyalty.transacoes
    GROUP BY ALL
) ,

Cliente_inicio AS (
    SELECT
        idCliente,
        MIN(DtDia) AS Dia_min
    FROM Clientes
    GROUP BY ALL
)

SELECT 
    -- t1.DtDia,
    SUBSTR(t1.DtDia, 0, 7) AS DtMes, 
    SUM(t1.qtdeTransacoes) AS qtdeTransacoes,
    COUNT(DISTINCT t1.idCliente) AS QtdeClientes,
    COUNT(DISTINCT t2.idCliente) AS novosUsuarios
FROM Clientes AS t1 
LEFT JOIN Cliente_inicio AS t2
    ON t1.DtDia = t2.Dia_min
GROUP BY ALL
ORDER BY DtMes --t1.DtDia


Databricks visualization. Run in Databricks to view.